# Bag of words and the vector-space model

Every document becomes a vector, a list of numbers. Once texts are numbers, we can compare and classify them at scale.

Bag of words (BoW): which words matters, but not the order.

Term-document matrix:
- Each word is a row
- Each document is a column
- Each cell says something about the presence of the word in that document:
  - raw term frequency: how many times the word appears;
  - binary BoW: 1 if the word appears at least once, 0 if not.

In [3]:
from pathlib import Path  # Provides tools for locating files and folders.
import pandas as pd  # Provides DataFrames for displaying the term-document matrices.
pd.set_option("display.max_columns", None)  # Shows every column instead of hiding some with "...".
from sklearn.feature_extraction.text import CountVectorizer  # Converts text into word-count features.

source = Path("../data/FITZ-TGG-25_paragraphs")  # Points to the folder containing the paragraph text files.
files = sorted(source.glob("*.txt"))  # Finds all .txt files in that folder and sorts their paths.
paragraphs = [path.read_text(encoding="utf-8").strip() for path in files]  # Reads each file into memory and removes whitespace at its edges; files remain unchanged.
names = [path.stem for path in files]  # Gets each filename without its .txt extension to use as a column label.

vec = CountVectorizer()  # Creates a word counter that lowercases text and splits it into tokens.
X = vec.fit_transform(paragraphs)  # Learns the vocabulary and counts each word in each paragraph; rows are paragraphs and columns are words.

tdm = pd.DataFrame(X.T.toarray(), index=vec.get_feature_names_out(), columns=names)  # Transposes the counts so words are rows and paragraph filenames are columns.

print(tdm.shape)  # Prints the matrix dimensions: (number of distinct words, number of paragraph files).

print(tdm) # Displays the term-document matrix with words as rows and paragraph filenames as columns.

(1656, 91)
           par01  par10  par11  par12  par13  par14  par15  par16  par17  \
1915           0      0      0      0      1      0      0      0      0   
abandoned      0      0      0      0      0      0      0      0      0   
able           0      0      0      0      0      0      0      0      0   
abnormal       0      1      0      0      0      0      0      0      0   
abortive       0      0      1      0      0      0      0      0      0   
...          ...    ...    ...    ...    ...    ...    ...    ...    ...   
york           0      0      0      0      0      0      0      0      0   
you            0      0      0      0      0      0      1      0      0   
young          0      1      0      0      0      1      0      0      0   
younger        0      0      0      0      0      0      0      0      0   
your           0      0      0      0      0      0      0      0      0   

           par18  par19  par20  par21  par22  par23  par24  par25  par26  \


In the paragraphs, there are both paragraphs from the published text and paragraphs from Fitzgerald's drafts. I want to find all paragraphs (drafts or published) close to the text of `par01.txt`:

    “Shall we take anything to drink?” called Daisy from an upper window.
    “I’ll get some whiskey,” answered Tom. He went inside.
    Gatsby turned to me rigidly:
    “I can’t say anything in his house, old sport.”
    “She’s got an indiscreet voice,” I remarked. “It’s full of—” I hesitated.
    “Her voice is full of money,” he said suddenly.


In [ ]:
print(tdm.loc[["voice", "money"]])  # Displays counts for these words; errors if any word is absent from the vocabulary.


         par01  par10  par11  par12  par13  par14  par15  par16  par17  par18  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par19  par20  par21  par22  par23  par24  par25  par26  par27  par28  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par29  par30  par31  par32  par33  par34  par35  par36  par37  par38  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par39  par40  par41  par42  par43  par44  par45  par46  par47  par48  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par49  par50  p

### Exercise
Look for articles on "Gatsby" and "race" in the [UB Catalogue](https://ubbern.swisscovery.slsp.ch). Try `Gatsby race` and `race Gatsby` and see if the results change.

Probably the same happens if you search your own emails or if you search in a website, for example in Wikipedia.

All these search engines are based on the presence of a word, independently from its meaning and order.

### Pros and cons of BoW
Pros
- Numerical basis for comparison -> analysis at scale becomes possible.
   
Cons
- No semantic ("gossip" =! "rumours", "arm" body = "arm" weapon)
- No order, no syntax: "Daisy shook her head decisively at Miss Baker" = "Miss Baker shook her head decisively at Daisy"
- Vectors are computationally difficult, because they are high dimensional and sparse (many `0`) --> Calculating similarity is cumbersome
- Not every word is equally important

In [28]:
# The word counts for "rumours" and "gossip" across all paragraphs.
# There is no way to see that they are close in meaning.
print(tdm.loc[["rumours", "gossip"]])  # Displays counts for these words; errors if any word is absent from the vocabulary.

# Displays the word-count vectors for par98.txt and par99.txt.
print(tdm[["par98_fake", "par99_fake"]])  
# Show only the rows where the word count is greater than zero for these paragraphs.
print(tdm.loc[(tdm["par98_fake"] > 0) | (tdm["par99_fake"] > 0), ["par98_fake", "par99_fake"]])



         par01  par10  par11  par12  par13  par14  par15  par16  par17  par18  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par19  par20  par21  par22  par23  par24  par25  par26  par27  par28  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par29  par30  par31  par32  par33  par34  par35  par36  par37  par38  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par39  par40  par41  par42  par43  par44  par45  par46  par47  par48  \
rumours      0      0      0      0      0      0      0      0      0      0   
gossip       0      0      0      0      0      0      0      0      0      0   

         par49  par50  p

# TF/IDF

If not every word is equally important (limitation of BoW) -> look for words that are **frequent** in a specific document but **rare** in the corpus. For example, stopwords are not significant.

Score
- **Term Frequency**. How often the world appears in a document. Number of occurrence, relative to the document's length.
- **Inverse Document Frequency**. How rare or common the word is. total number of documents ÷ number of documents containing the word. We take the logarithm of this to tame the huge range of values: word frequencies follow Zipf's law (a few words are extremely frequent, most are very rare).

Example: In a corpus of documents about literary studies, the word "autodiegetic" appears 30 times in document X and few times in other documents. The word 'autodiegetic' is thus particularly relevant to categorise document X compared to the others.

Calculate TF/IDF for the word "voice" in our corpus of paragraphs.


In [ ]:
import math  # Provides the natural logarithm used in the IDF calculation.
import spacy  # Imports SpaCy.

nlp = spacy.load("en_core_web_sm")  # Loads the small English model; it must be installed in this environment.

# Tokenizes each paragraph and keeps alphabetic words longer than one letter.
tokens_by_paragraph = [[token for token in nlp(text) if token.is_alpha and len(token.text) > 1] for text in paragraphs]

# Sets the word whose TF-IDF score to calculate.
word = "voice"

# Counts occurrences of the word in each paragraph.
counts = [sum(token.text.lower() == word for token in tokens) for tokens in tokens_by_paragraph]  

# Counts retained word tokens in each paragraph.
lengths = [len(tokens) for tokens in tokens_by_paragraph]  

# Counts paragraphs containing the word.
df = sum(count > 0 for count in counts)  

# Calculates IDF, or zero if the word is absent from all paragraphs.
idf = math.log(len(paragraphs) / df) if df else 0  

for name, count, length in zip(names, counts, lengths):  # Pairs each paragraph name with its count and length.
    tf = count / length if length else 0  # Calculates relative frequency, avoiding division by zero.
    print(f"{name}: tf={tf:.4f}  idf={idf:.3f}  tf-idf={tf * idf:.5f}")  # Prints the scores for this paragraph.

par01: tf=0.0385  idf=2.113  tf-idf=0.08127
par10: tf=0.0000  idf=2.113  tf-idf=0.00000
par11: tf=0.0000  idf=2.113  tf-idf=0.00000
par12: tf=0.0000  idf=2.113  tf-idf=0.00000
par13: tf=0.0000  idf=2.113  tf-idf=0.00000
par14: tf=0.0000  idf=2.113  tf-idf=0.00000
par15: tf=0.0000  idf=2.113  tf-idf=0.00000
par16: tf=0.0000  idf=2.113  tf-idf=0.00000
par17: tf=0.0000  idf=2.113  tf-idf=0.00000
par18: tf=0.0000  idf=2.113  tf-idf=0.00000
par19: tf=0.0000  idf=2.113  tf-idf=0.00000
par20: tf=0.0000  idf=2.113  tf-idf=0.00000
par21: tf=0.0000  idf=2.113  tf-idf=0.00000
par22: tf=0.0000  idf=2.113  tf-idf=0.00000
par23: tf=0.0000  idf=2.113  tf-idf=0.00000
par24: tf=0.0000  idf=2.113  tf-idf=0.00000
par25: tf=0.0000  idf=2.113  tf-idf=0.00000
par26: tf=0.0256  idf=2.113  tf-idf=0.05418
par27: tf=0.0000  idf=2.113  tf-idf=0.00000
par28: tf=0.0000  idf=2.113  tf-idf=0.00000
par29: tf=0.0000  idf=2.113  tf-idf=0.00000
par30: tf=0.0000  idf=2.113  tf-idf=0.00000
par31: tf=0.0000  idf=2.113  tf-

### Exercise 

Calculate TF/IDF for "Daisy" across paragraphs.

Create a cell here below and copy paste the code from the previous cell. Change "voice" to "Daisy".

par01: tf=0.0000  idf=0.000  tf-idf=0.00000
par10: tf=0.0000  idf=0.000  tf-idf=0.00000
par11: tf=0.0000  idf=0.000  tf-idf=0.00000
par12: tf=0.0000  idf=0.000  tf-idf=0.00000
par13: tf=0.0000  idf=0.000  tf-idf=0.00000
par14: tf=0.0000  idf=0.000  tf-idf=0.00000
par15: tf=0.0000  idf=0.000  tf-idf=0.00000
par16: tf=0.0000  idf=0.000  tf-idf=0.00000
par17: tf=0.0000  idf=0.000  tf-idf=0.00000
par18: tf=0.0000  idf=0.000  tf-idf=0.00000
par19: tf=0.0000  idf=0.000  tf-idf=0.00000
par20: tf=0.0000  idf=0.000  tf-idf=0.00000
par21: tf=0.0000  idf=0.000  tf-idf=0.00000
par22: tf=0.0000  idf=0.000  tf-idf=0.00000
par23: tf=0.0000  idf=0.000  tf-idf=0.00000
par24: tf=0.0000  idf=0.000  tf-idf=0.00000
par25: tf=0.0000  idf=0.000  tf-idf=0.00000
par26: tf=0.0000  idf=0.000  tf-idf=0.00000
par27: tf=0.0000  idf=0.000  tf-idf=0.00000
par28: tf=0.0000  idf=0.000  tf-idf=0.00000
par29: tf=0.0000  idf=0.000  tf-idf=0.00000
par30: tf=0.0000  idf=0.000  tf-idf=0.00000
par31: tf=0.0000  idf=0.000  tf-

### Use `scikit-learn`

Instead of calculating TF/IDF manually as done here above, we can use the Python package `scikit-learn`.

Note: `scikit-learn` uses a slightly different IDF to avoid zeros, so its numbers differ a little from the hand-made ones. The ranking logic is the same.


In [ ]:
# Calculate TF/IDF of "voice" in each paragraph
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")   # also drop English stopwords
T = tfidf.fit_transform(paragraphs)
terms = tfidf.get_feature_names_out()

word = "voice"  # Try with Daisy

for i, name in enumerate(names):
    row = T[i].toarray().ravel()
    if word in terms:
        idx = terms.tolist().index(word)
        tfidf_score = row[idx]
    else:
        tfidf_score = 0
    print(f"{name}: tf-idf={tfidf_score:.5f}")

par01: tf-idf=0.00000
par10: tf-idf=0.00000
par11: tf-idf=0.00000
par12: tf-idf=0.00000
par13: tf-idf=0.00000
par14: tf-idf=0.00000
par15: tf-idf=0.00000
par16: tf-idf=0.00000
par17: tf-idf=0.00000
par18: tf-idf=0.00000
par19: tf-idf=0.00000
par20: tf-idf=0.00000
par21: tf-idf=0.00000
par22: tf-idf=0.00000
par23: tf-idf=0.00000
par24: tf-idf=0.00000
par25: tf-idf=0.00000
par26: tf-idf=0.00000
par27: tf-idf=0.00000
par28: tf-idf=0.00000
par29: tf-idf=0.00000
par30: tf-idf=0.00000
par31: tf-idf=0.00000
par32: tf-idf=0.00000
par33: tf-idf=0.00000
par34: tf-idf=0.00000
par35: tf-idf=0.00000
par36: tf-idf=0.00000
par37: tf-idf=0.00000
par38: tf-idf=0.00000
par39: tf-idf=0.00000
par40: tf-idf=0.00000
par41: tf-idf=0.00000
par42: tf-idf=0.00000
par43: tf-idf=0.00000
par44: tf-idf=0.00000
par45: tf-idf=0.00000
par46: tf-idf=0.00000
par47: tf-idf=0.00000
par48: tf-idf=0.00000
par49: tf-idf=0.00000
par50: tf-idf=0.00000
par51: tf-idf=0.00000
par52: tf-idf=0.00000
par53: tf-idf=0.00000
par54: tf-

Calculate the 3 most characteristic words of each paragraph.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")   # also drop English stopwords
T = tfidf.fit_transform(paragraphs)
terms = tfidf.get_feature_names_out()

for i, name in enumerate(names):
    row = T[i].toarray().ravel()
    top = row.argsort()[::-1][:3]                 # indices of the 8 highest scores
    print(name, "->", ", ".join(terms[top]))

par01 -> voice, indiscreet, sport
par10 -> snobbishly, judgements, intimate
par11 -> gatsby, founded, wanted
par12 -> carraways, carries, grandfather
par13 -> father, supposed, bond
par14 -> finnish, country, house
par15 -> road, recently, village
par16 -> original, conferred, pathfinder
par17 -> bursts, fast, growing
par18 -> life, specialists, successfully
par19 -> island, columbus, pair
par20 -> eyesore, view, mansion
par21 -> glittered, begins, history
par22 -> wealthy, national, ends
par23 -> seeking, irrecoverable, dramatic
par24 -> windy, warm, house
par25 -> body, enormous, capable
par26 -> tenor, paternal, gruff
par27 -> harsh, final, stronger
par28 -> sunny, sunken, vista
par29 -> blew, coloured, windows
par30 -> women, curtains, young
par31 -> extended, hint, fall
par32 -> laughed, forward, laugh
par33 -> murmur, paralysed, surname
par34 -> lips, imperceptibly, self
par35 -> bright, things, voice
par36 -> ecstatically, sent, cried
par37 -> painted, wail, wreath
par38 -> irre

### In practice
For example...
- Ranking system for information retrieval: Search for three words in a corpus = calculate their combined TF/IDF score for those words in every document. 
- Spam filters. Words like "winner", "free", "guarantee" appear often in spam but rarely in normal emails.
- Document tagging.


### Pros and Cons
Pros:
- highly interpretable
- computational efficient

Cons (as BoW):
- No syntax
- No semantic